**Connect to PostgreSQL**

A connection is established with the `CHOCO_CRUNCH` PostgreSQL database using `psycopg2`.

A cursor is created to execute SQL statements from Python. This connection will be used to create the database tables and insert the feature-engineered data.

In [1]:
import os
import psycopg2
from dotenv import load_dotenv

load_dotenv()

conn = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

# Create a cursor object
cur = conn.cursor()

print("Connected to CHOCO_CRUNCH successfully!")

Connected to CHOCO_CRUNCH successfully!


**product_info**

**1) Count products per brand**

In [2]:
query = """
SELECT
    brand,
    COUNT(*) AS product_count
FROM product_info
WHERE brand <> 'Missing'
GROUP BY brand
ORDER BY product_count DESC;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

4028


[('Hacendado', 353),
 ('Tesco', 267),
 ('Carrefour', 148),
 ('Bjorg', 99),
 ('Heinz', 96),
 ('Lidl', 90),
 ('Nestlé', 87),
 ("Sainsbury's", 79),
 ('Aldi', 75),
 ('Gerblé', 72)]

**2)Count unique products per brand**

In [3]:
query = """
SELECT
    brand,
    COUNT(DISTINCT product_code) AS unique_product_count
FROM product_info
WHERE brand <> 'Missing'
GROUP BY brand
ORDER BY unique_product_count DESC;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

4028


[('Hacendado', 353),
 ('Tesco', 267),
 ('Carrefour', 148),
 ('Bjorg', 99),
 ('Heinz', 96),
 ('Lidl', 90),
 ('Nestlé', 87),
 ("Sainsbury's", 79),
 ('Aldi', 75),
 ('Gerblé', 72)]

**3)Top 5 brands by product count**

In [4]:
query = """
SELECT
    brand,
    COUNT(*) AS product_count
FROM product_info
WHERE brand <> 'Missing'
GROUP BY brand
ORDER BY product_count DESC
LIMIT 5;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

5


[('Hacendado', 353),
 ('Tesco', 267),
 ('Carrefour', 148),
 ('Bjorg', 99),
 ('Heinz', 96)]

**4)Products with missing product name**

In [5]:
query = """
SELECT
    product_code,
    product_name,
    brand
FROM product_info
WHERE product_name = 'Missing';
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

306


[('6111242100985', 'Missing', 'Jaouda'),
 ('6111099003897', 'Missing', 'lilia'),
 ('6111128000460', 'Missing', 'aïn Saiss'),
 ('20005733', 'Missing', 'Alesto,Lidl'),
 ('8425197712024', 'Missing', 'Maruja'),
 ('6111035003523', 'Missing', 'BAHIA'),
 ('20047238', 'Missing', 'Alesto,Alesto Lidl,Alesto Selection,Lidl'),
 ('6111032008583', 'Missing', 'Gervais'),
 ('6111003031107', 'Missing', '8dh'),
 ('6111184001197', 'Missing', 'Star')]

**5)Number of unique brands**

In [6]:
query = """
SELECT
    COUNT(DISTINCT brand) AS unique_brand_count
FROM product_info
WHERE brand <> 'Missing';
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

1


[(4028,)]

**6) Products with code starting with '3'**

In [7]:
query = """
SELECT
    product_code,
    product_name,
    brand
FROM product_info
WHERE product_code LIKE '3%';
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

3673


[('3274080005003', 'Eau De Source', 'Cristaline'),
 ('3046920029759', 'Edelbitterschokolade Mild 90%', 'Lindt&Sprüngli'),
 ('3046920028004', 'Excellence 70% Cocoa Intense Dark', 'Lindt'),
 ('3017620425035', 'Nutella', 'Ferrero'),
 ('3175680011480', 'Sésame', 'Gerblé'),
 ('3046920028363', 'Excellence 85% Cacao Rich Dark', 'Lindt'),
 ('3268840001008', 'CRISTALINE Eau De Source 0.5L', 'Cristaline'),
 ('3017620422003', 'Nutella', 'Ferrero'),
 ('3362600011044', 'Henry’s', "Henry's"),
 ('3362600011228', 'Sable coco Henry s 42g', "Henry's")]

**nutrient_info**

**1)Top 10 products with highest energy-kcal_value**

In [8]:
query = """
SELECT
    p.product_code,
    p.product_name,
    p.brand,
    n.energy_kcal_value
FROM product_info p
JOIN nutrient_info n
    ON p.product_code = n.product_code
ORDER BY n.energy_kcal_value DESC
LIMIT 10;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

10


[('8480000054647', 'Chía', 'Hacendado', 45793.499043977),
 ('5010029221701', 'PROTEIN', 'Weetabix', 14770.5544933078),
 ('6111259090132', 'Pizzarella premium', 'Gastro mixte', 5280.0),
 ('3088543506255', "Sirop d'Agave", 'Sunny Via', 1600.0),
 ('36000291452', 'penaut butter creamy', 'Alaska', 904.0),
 ('6111024001530', 'Huile Lesieur 3 Graines', 'Lesieur', 900.0),
 ('6111024004746', "Huile d'olive vierge de Maroc", 'ALHORRA', 900.0),
 ('6111099000254', "Huile d'olive", 'oued souss', 900.0),
 ('6111099000599', 'Missing', 'lio', 900.0),
 ('6111024002186', 'Houilor', 'Huilor', 900.0)]

**2)Average sugars_value per nova-group**

In [9]:
query = """
SELECT
    nova_group,
    AVG(sugars_value) AS average_sugar
FROM nutrient_info
GROUP BY nova_group
ORDER BY nova_group;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

4


[(1, 6.007658361462918),
 (2, 17.15793877551024),
 (3, 6.8660973963445295),
 (4, 12.866388431766797)]

**3)Count products with fat_value > 20g**

In [10]:
query = """
SELECT
    COUNT(*) AS product_count
FROM nutrient_info
WHERE fat_value > 20;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

1


[(2914,)]

**4)Average carbohydrates_value per product**

In [11]:
query = '''
SELECT
    product_code,
    AVG(carbohydrates_value) AS average_carbohydrates
FROM nutrient_info
GROUP BY product_code
ORDER BY product_code;
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

11783


[('10001295', 65.0),
 ('10001417', 5.6),
 ('10022708', 8.3),
 ('10023594', 5.3),
 ('1002553', 13.1),
 ('1002942', 69.6),
 ('1005912', 1.2),
 ('10061282', 11.9),
 ('10065891', 11.9),
 ('10065907', 11.9)]

**5)Products with sodium_value > 1g**

In [12]:
query = '''
SELECT
    p.product_code,
    p.product_name,
    p.brand,
    n.sodium_value
FROM product_info AS p
JOIN nutrient_info AS n
    ON p.product_code = n.product_code
WHERE n.sodium_value > 1
ORDER BY n.sodium_value DESC;
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

660


[('3445850024023', 'Sel moulu de Guérande', 'Le Guérandais', 34000.0),
 ('8712566351022', "Moutarde à l'Ancienne", 'Maille, Unilever', 5000.0),
 ('9352042000328', 'Vegemite', 'Bega', 3300.0),
 ('8853662056005', 'Sriracha', 'Flying Goose Brand', 2920.0),
 ('3036810201280', 'Dijon Originale', 'MAILLE', 2000.0),
 ('8801073110502', 'Buldak HOT Chicken Flavor Ramen', 'Samyang', 1630.0),
 ('3155250361825',
  'President spreadable lightly salted',
  'Lactalis, Président',
  1300.0),
 ('6111005054081', 'Levure chimique en poudre', 'alsa', 1287.0),
 ('20034658', 'Saumon fumé', 'nautica', 1265.0),
 ('8480000188175', 'Salmón al natural', 'HACENDADO', 1200.0)]

**6)Count products with non-zero fruits-vegetables-nuts content**

In [13]:
query = '''
SELECT
    COUNT(*) AS product_count
FROM nutrient_info
WHERE fruits_vegetables_nuts_estimate_from_ingredients_100g > 0;
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

1


[(6342,)]

**7)Products with energy-kcal_value > 500**

In [14]:
query = '''
SELECT
    p.product_code,
    p.product_name,
    p.brand,
    n.energy_kcal_value
FROM product_info AS p
JOIN nutrient_info AS n
    ON p.product_code = n.product_code
WHERE n.energy_kcal_value > 500
ORDER BY n.energy_kcal_value DESC;
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

1736


[('8480000054647', 'Chía', 'Hacendado', 45793.499043977),
 ('5010029221701', 'PROTEIN', 'Weetabix', 14770.5544933078),
 ('6111259090132', 'Pizzarella premium', 'Gastro mixte', 5280.0),
 ('3088543506255', "Sirop d'Agave", 'Sunny Via', 1600.0),
 ('36000291452', 'penaut butter creamy', 'Alaska', 904.0),
 ('3596710375561',
  "Huile d'olive vierge extra Assemblage d'huiles d'olive origine UE et non UE",
  'Auchan, Auchan Bio',
  900.0),
 ('6111024004746', "Huile d'olive vierge de Maroc", 'ALHORRA', 900.0),
 ('3256221712155', "Huile d'olive vierge extra Bouteille 1L", 'U', 900.0),
 ('8901088000451', 'Parachnute 100% Pure Coconut Oil', 'Parachute', 900.0),
 ('3250390002659', "Hulie d'olive vierge extra", "Bouton d'Or", 900.0)]

**Derived_metrics**

**1)Count products per calorie_category**

In [15]:
query = '''
SELECT
    calorie_category,
    COUNT(*) AS product_count
FROM derived_metrics
GROUP BY calorie_category
ORDER BY product_count DESC;
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

3


[('High', 6812), ('Low', 3048), ('Moderate', 1923)]

**2)Count of High Sugar products**

In [16]:
query = '''
SELECT
    COUNT(*) AS high_sugar_count
FROM derived_metrics
WHERE sugar_category = 'High Sugar';
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

1


[(3263,)]

**3)Average sugar_to_carb_ratio for High Calorie products**

In [17]:
query = """
SELECT
    AVG(sugar_to_carb_ratio) AS average_ratio
FROM derived_metrics
WHERE calorie_category = 'High Calorie'
  AND sugar_to_carb_ratio NOT IN ('Infinity', '-Infinity');
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

1


[(None,)]

**4)Products that are both High Calorie and High Sugar**

In [18]:
query = '''
SELECT
    product_code,
    calorie_category,
    sugar_category
FROM derived_metrics
WHERE calorie_category = 'High'
  AND sugar_category = 'High Sugar';
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

2809


[('6111035002175', 'High', 'High Sugar'),
 ('6111035000058', 'High', 'High Sugar'),
 ('7622210449283', 'High', 'High Sugar'),
 ('3046920028004', 'High', 'High Sugar'),
 ('6111035003035', 'High', 'High Sugar'),
 ('3017620425035', 'High', 'High Sugar'),
 ('3175680011480', 'High', 'High Sugar'),
 ('6111162000839', 'High', 'High Sugar'),
 ('6111035001659', 'High', 'High Sugar'),
 ('6111128000460', 'High', 'High Sugar')]

**5) Number of products marked as ultra-processed**

In [19]:
query = '''
SELECT
    COUNT(*) AS ultra_processed_count
FROM derived_metrics
WHERE is_ultra_processed = 'Yes';
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

1


[(6298,)]

**6) Products with sugar_to_carb_ratio > 0.7**

In [20]:
query = """
SELECT
    product_code,
    sugar_to_carb_ratio
FROM derived_metrics
WHERE sugar_to_carb_ratio > 0.7
  AND sugar_to_carb_ratio NOT IN ('Infinity', '-Infinity')
ORDER BY sugar_to_carb_ratio DESC;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

4076


[('20139315', 16.0),
 ('20139346', 10.0),
 ('6111259344129', 6.682926829268293),
 ('6111184000244', 4.3742857142857146),
 ('3456700013005', 4.0),
 ('6291003084881', 2.6646706586826348),
 ('20465384', 2.0),
 ('5449000147417', 1.8279569892473118),
 ('8901030921797', 1.7806451612903227),
 ('5014276701504', 1.25)]

**7)Average sugar_to_carb_ratio per calorie_category**

In [21]:
query = """
SELECT
    calorie_category,
    AVG(sugar_to_carb_ratio) AS average_ratio
FROM derived_metrics
WHERE sugar_to_carb_ratio NOT IN ('Infinity', '-Infinity')
GROUP BY calorie_category
ORDER BY average_ratio DESC;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

3


[('Low', 0.6075408139356088),
 ('Moderate', 0.44648548940058047),
 ('High', 0.38180833458362873)]

**Join Queries**

**1)Top 5 brands with most High Calorie products**

In [22]:
query = """
SELECT
    p.brand,
    COUNT(*) AS high_calorie_count
FROM product_info p
JOIN derived_metrics d
    ON p.product_code = d.product_code
WHERE d.calorie_category = 'High Calorie'
  AND p.brand <> 'Missing'
GROUP BY p.brand
ORDER BY high_calorie_count DESC
LIMIT 5;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

0


[]

**2)Average energy-kcal_value for each calorie_category**

In [23]:
query = '''
SELECT
    d.calorie_category,
    AVG(n.energy_kcal_value) AS average_energy_kcal
FROM derived_metrics AS d
JOIN nutrient_info AS n
    ON d.product_code = n.product_code
GROUP BY d.calorie_category
ORDER BY d.calorie_category;
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

3


[('High', 420.826369617224),
 ('Low', 48.374878201385336),
 ('Moderate', 143.17141621233276)]

**3)Count of ultra-processed products per brand**

In [24]:
query = """
SELECT
    p.brand,
    COUNT(*) AS ultra_processed_count
FROM product_info p
JOIN derived_metrics d
    ON p.product_code = d.product_code
WHERE d.is_ultra_processed = 'Yes'
  AND p.brand <> 'Missing'
GROUP BY p.brand
ORDER BY ultra_processed_count DESC;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

2371


[('Hacendado', 195),
 ('Tesco', 92),
 ('Carrefour', 69),
 ('Gerblé', 67),
 ('Bonne Maman', 64),
 ('Bjorg', 59),
 ('Fleury Michon', 59),
 ('Coca-Cola', 51),
 ('Nestlé', 50),
 ('Heinz', 50)]

**4)Products with High Sugar and High Calorie along with brand**

In [25]:
query = """
SELECT
    p.product_code,
    p.product_name,
    p.brand,
    d.calorie_category,
    d.sugar_category
FROM product_info p
JOIN derived_metrics d
    ON p.product_code = d.product_code
WHERE d.calorie_category = 'High Calorie'
  AND d.sugar_category = 'High Sugar'
  AND p.brand <> 'Missing';
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

0


[]

**5)Average sugar content per brand for ultra-processed products**

In [26]:
query = '''
SELECT
    p.brand,
    AVG(n.sugars_value) AS average_sugar_content
FROM product_info AS p
JOIN nutrient_info AS n
    ON p.product_code = n.product_code
JOIN derived_metrics AS d
    ON p.product_code = d.product_code
WHERE d.is_ultra_processed = 'Yes'
AND p.brand <> 'Missing'
GROUP BY p.brand
ORDER BY average_sugar_content DESC;
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results[:10]

2371


[('Saint Louis Sucre, Saint-Louis, Tutti free', 99.0),
 ('Vichy', 98.5),
 ('Ferrero,Mixte,TIC TAC', 94.5),
 ('tictac', 94.5),
 ('Ferrero, TIC TAC', 90.7),
 ('Canderel, Merisant', 90.0),
 ('Moulin de Valdonne', 81.5),
 ('Abram Lyle & Sons', 79.0),
 ('La maison guiot', 78.7),
 ("Lyle's", 77.5)]

**6)Number of products with fruits/vegetables/nuts content in each calorie_category**

In [27]:
query = '''
SELECT
    d.calorie_category,
    COUNT(*) AS product_count
FROM derived_metrics AS d
JOIN nutrient_info AS n
    ON d.product_code = n.product_code
WHERE n.fruits_vegetables_nuts_estimate_from_ingredients_100g > 0
GROUP BY d.calorie_category
ORDER BY product_count DESC;
'''

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

3


[('High', 3483), ('Low', 1726), ('Moderate', 1133)]

**7)Top 5 products by sugar_to_carb_ratio with their calorie and sugar category**

In [28]:
query = """
SELECT
    p.product_code,
    p.product_name,
    p.brand,
    d.sugar_to_carb_ratio,
    d.calorie_category,
    d.sugar_category
FROM product_info p
JOIN derived_metrics d
    ON p.product_code = d.product_code
WHERE d.sugar_to_carb_ratio NOT IN ('Infinity', '-Infinity')
ORDER BY d.sugar_to_carb_ratio DESC
LIMIT 5;
"""

cur.execute(query)
results = cur.fetchall()

print(len(results))
results

5


[('20139315',
  'Beurre Gastronomique Doux',
  'Missing',
  16.0,
  'High',
  'Low Sugar'),
 ('20139346',
  'Beurre Doux Extrafin',
  'Envia / Lidl',
  10.0,
  'High',
  'Low Sugar'),
 ('6111259344129',
  'Sergio',
  'excelo',
  6.682926829268293,
  'High',
  'High Sugar'),
 ('6111184000244',
  'El Baraka',
  'vmm',
  4.3742857142857146,
  'High',
  'High Sugar'),
 ('3456700013005',
  'Truite des Pyrénées fumée',
  'Pêcheries Basques',
  4.0,
  'Moderate',
  'Low Sugar')]